In [ ]:
import json, os

CHECKPOINT_DIR = "/kaggle/working/cv_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

fold2_metrics = {
    "acc" : 0.9087,
    "prec": 0.9101,
    "rec" : 0.9087,
    "f1"  : 0.9089,
    "auc" : 0.9626
}

path = os.path.join(CHECKPOINT_DIR, "fold_2.json")
with open(path, "w") as f:
    json.dump({"fold": 2, "metrics": fold2_metrics}, f)
print(f"Fold 2 saved: {path}")

Fold 2 saved: /kaggle/working/cv_checkpoints/fold_2.json


In [ ]:
import gc
import json
import os
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel, AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    roc_auc_score, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

# ============================================================
# CHECKPOINT CONFIG
# ============================================================

# Fold results are saved here after each fold completes
CHECKPOINT_DIR = "/kaggle/working/cv_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# If you attach previous session's output as input, put its path here
# Kaggle Notebook → Input → Add Output → select your previous run
PREV_OUTPUT_DIR = "/kaggle/input/datasets/mdistiyakhasanmaruf/cv-checkpoints-new"

# Pull fold results from previous session (if attached)
if os.path.exists(PREV_OUTPUT_DIR):
    pulled = 0
    for fname in os.listdir(PREV_OUTPUT_DIR):
        if fname.startswith("fold_") and fname.endswith(".json"):
            src = os.path.join(PREV_OUTPUT_DIR, fname)
            dst = os.path.join(CHECKPOINT_DIR, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Resumed from previous session: {fname}")
                pulled += 1
    if pulled == 0:
        print("  Previous output found but nothing new to pull.")
else:
    print("  No previous session output found — starting fresh.")


def fold_already_done(fold):
    """Check if this fold's result JSON already exists."""
    return os.path.exists(os.path.join(PREV_OUTPUT_DIR, f"fold_{fold}.json"))


def save_fold_result(fold, metrics):
    """Save fold metrics to disk so it persists after commit."""
    path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}.json")
    with open(path, "w") as f:
        json.dump({"fold": fold, "metrics": metrics}, f)
    print(f"  Fold {fold} result saved → {path}")


def load_fold_result(fold):
    """Load a previously saved fold result. Returns None if not found."""
    path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None


# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("/kaggle/input/datasets/mdistiyakhasanmaruf/bilingual-hate-speech/spelling_correct_with_removeNumber - spelling_correct_with_removeNumber.csv")
df.columns     = ["sentence", "labels"]
df["sentence"] = df["sentence"].astype(str)
df["labels"]   = df["labels"].astype(int)

print(f"Dataset size: {len(df)}")
print(df["labels"].value_counts())

texts       = df["sentence"].tolist()
labels      = df["labels"].tolist()
num_classes = 2

# ============================================================
# TOKENIZERS — loaded once, reused across folds
# ============================================================

print("\nLoading tokenizers...")
bert_tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer    = AutoTokenizer.from_pretrained("xlm-roberta-base")
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
MAX_LEN = 128
print("Tokenizers loaded.")

# ============================================================
# DATASETS
# ============================================================

class EnsembleDataset(Dataset):
    def __init__(self, texts, labels, bert_tok, xlmr_tok, deberta_tok, max_len=128):
        self.texts       = texts
        self.labels      = labels
        self.bert_tok    = bert_tok
        self.xlmr_tok    = xlmr_tok
        self.deberta_tok = deberta_tok
        self.max_len     = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        be = self.bert_tok(   text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        xe = self.xlmr_tok(   text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        de = self.deberta_tok(text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "bert_ids"    : be["input_ids"].squeeze(0),
            "bert_mask"   : be["attention_mask"].squeeze(0),
            "xlmr_ids"    : xe["input_ids"].squeeze(0),
            "xlmr_mask"   : xe["attention_mask"].squeeze(0),
            "deberta_ids" : de["input_ids"].squeeze(0),
            "deberta_mask": de["attention_mask"].squeeze(0),
            "label"       : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


class StudentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


class KDDataset(Dataset):
    def __init__(self, student_ds, teacher_logits):
        self.student_ds    = student_ds
        self.teacher_logits = teacher_logits

    def __len__(self):
        return len(self.student_ds)

    def __getitem__(self, idx):
        item = self.student_ds[idx]
        item["teacher_logits"] = self.teacher_logits[idx]
        return item

# ============================================================
# LOSS FUNCTIONS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features    = F.normalize(features, p=2, dim=1)
        sim         = torch.matmul(features, features.T) / self.temperature
        labels      = labels.contiguous().view(-1, 1)
        mask        = torch.eq(labels, labels.T).float().to(features.device)
        mask_no_self = mask - torch.eye(labels.shape[0]).to(features.device)
        exp_sim     = torch.exp(sim) * (1 - torch.eye(labels.shape[0]).to(features.device))
        log_prob    = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        mask_sum    = mask_no_self.sum(dim=1).clamp(min=1)
        return -(mask_no_self * log_prob).sum(dim=1).div(mask_sum).mean()


class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return grad.neg() * ctx.alpha, None


class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)

# ============================================================
# TEACHER MODEL
# ============================================================

class DynamicFusionNet(nn.Module):
    def __init__(self, num_classes=2, num_languages=2):
        super().__init__()
        self.bert    = AutoModel.from_pretrained("bert-base-uncased")
        self.xlmr    = AutoModel.from_pretrained("xlm-roberta-base")
        self.deberta = AutoModel.from_pretrained("microsoft/deberta-base")

        for m in [self.bert, self.xlmr, self.deberta]:
            for p in m.embeddings.parameters():
                p.requires_grad = False
            for l in m.encoder.layer[:4]:
                for p in l.parameters():
                    p.requires_grad = False

        h = 768
        self.attention  = nn.MultiheadAttention(embed_dim=h, num_heads=8, batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(h, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
        self.grl             = GradientReversalLayer(alpha=1.0)
        self.lang_classifier = nn.Linear(h, num_languages)

    def forward(self, bert_ids, bert_mask, xlmr_ids, xlmr_mask, deb_ids, deb_mask):
        bc = self.bert(   input_ids=bert_ids,  attention_mask=bert_mask ).last_hidden_state[:, 0, :]
        xc = self.xlmr(   input_ids=xlmr_ids,  attention_mask=xlmr_mask).last_hidden_state[:, 0, :]
        dc = self.deberta(input_ids=deb_ids,   attention_mask=deb_mask  ).last_hidden_state[:, 0, :]
        st       = torch.stack((bc, xc, dc), dim=1)
        ao, _    = self.attention(st, st, st)
        fused    = torch.mean(ao, dim=1)
        return self.classifier(fused), self.lang_classifier(self.grl(fused)), fused

# ============================================================
# KD LOSS
# ============================================================

def kd_loss(student_logits, teacher_logits, hard_labels,
            alpha=0.5, temperature=4.0, weight=None):
    ce     = F.cross_entropy(student_logits, hard_labels, weight=weight)
    s_soft = F.log_softmax(student_logits / temperature, dim=-1)
    t_soft = F.softmax(teacher_logits / temperature,     dim=-1)
    kl     = F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature ** 2)
    return alpha * ce + (1 - alpha) * kl

# ============================================================
# 5-FOLD CV SETTINGS
# ============================================================

N_SPLITS         = 5
TEACHER_EPOCHS   = 10
TEACHER_PATIENCE = 3
TEACHER_LR       = 1e-5
ACCUM_STEPS      = 4

KD_EPOCHS        = 5
KD_PATIENCE      = 3
KD_LR            = 2e-5

lambda_scl = 0.05
lambda_grl = 0.02

skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
all_splits = list(skf.split(texts, labels))   # fixed splits — same every run

# Show resume status
already_done = [f for f in range(1, N_SPLITS + 1) if fold_already_done(f)]
remaining    = [f for f in range(1, N_SPLITS + 1) if not fold_already_done(f)]
print(f"\n  Already completed folds : {already_done if already_done else 'None'}")
print(f"  Remaining folds         : {remaining    if remaining    else 'None — all done!'}")

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION WITH RESUME SUPPORT")
print("Each fold: Fresh Teacher → Soft Labels → KD Student → Evaluate")
print("=" * 70)

# ============================================================
# FOLD LOOP
# ============================================================

for fold in range(1, N_SPLITS + 1):

    print(f"\n{'=' * 70}")
    print(f"FOLD {fold}/{N_SPLITS}")

    # Skip this fold if already completed in a previous session
    if fold_already_done(fold):
        result = load_fold_result(fold)
        m = result["metrics"]
        print(f"  Already completed — skipping.")
        print(f"  Saved result → Acc: {m['acc']*100:.2f}%  F1: {m['f1']*100:.2f}%  AUC: {m['auc']:.4f}")
        print(f"{'=' * 70}")
        continue

    print(f"{'=' * 70}")

    set_seed(42)

    train_idx, val_idx = all_splits[fold - 1]
    print(f"  Train: {len(train_idx)} | Val: {len(val_idx)}")

    fold_train_texts  = [texts[i]  for i in train_idx]
    fold_train_labels = [labels[i] for i in train_idx]
    fold_val_texts    = [texts[i]  for i in val_idx]
    fold_val_labels   = [labels[i] for i in val_idx]

    # Class weights
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=fold_train_labels)
    cw = torch.tensor(cw, dtype=torch.float).to(device)

    # Datasets
    train_ens_ds = EnsembleDataset(fold_train_texts, fold_train_labels,
                                   bert_tokenizer, xlmr_tokenizer, deberta_tokenizer, MAX_LEN)
    val_ens_ds   = EnsembleDataset(fold_val_texts, fold_val_labels,
                                   bert_tokenizer, xlmr_tokenizer, deberta_tokenizer, MAX_LEN)

    train_ens_ld = DataLoader(train_ens_ds, batch_size=32, shuffle=True,
                              worker_init_fn=lambda wid: set_seed(42 + wid))
    val_ens_ld   = DataLoader(val_ens_ds, batch_size=32)

    # ── STEP 1: Train Teacher ─────────────────────────────────────────
    print(f"\n[Fold {fold}] STEP 1: Training Teacher (DynamicFusionNet)...")

    torch.cuda.empty_cache(); gc.collect()
    teacher = DynamicFusionNet(num_classes=num_classes).to(device)

    focal  = FocalLoss(weight=cw, gamma=2.0)
    scl    = SupervisedContrastiveLoss(temperature=0.1).to(device)
    lang_c = nn.CrossEntropyLoss().to(device)

    t_opt         = torch.optim.AdamW(teacher.parameters(), lr=TEACHER_LR)
    t_total_steps = (len(train_ens_ld) // ACCUM_STEPS) * TEACHER_EPOCHS
    t_sched       = get_linear_schedule_with_warmup(t_opt,
                        num_warmup_steps=int(0.1 * t_total_steps),
                        num_training_steps=t_total_steps)

    best_t_acc   = 0.0
    t_patience   = 0
    TEACHER_PATH = f"/kaggle/working/fold{fold}_teacher.pt"

    for epoch in range(TEACHER_EPOCHS):
        teacher.train(); correct = total = 0; t_opt.zero_grad()

        for i, batch in enumerate(tqdm(train_ens_ld, desc=f"T-Ep{epoch+1}")):
            bi = batch["bert_ids"].to(device);    bm = batch["bert_mask"].to(device)
            xi = batch["xlmr_ids"].to(device);    xm = batch["xlmr_mask"].to(device)
            di = batch["deberta_ids"].to(device);  dm = batch["deberta_mask"].to(device)
            lb = batch["label"].to(device)
            dl = torch.randint(0, 2, (lb.size(0),)).to(device)

            logits, lang_l, fused = teacher(bi, bm, xi, xm, di, dm)
            loss = (focal(logits, lb)
                    + lambda_scl * scl(fused, lb)
                    + lambda_grl * lang_c(lang_l, dl))
            (loss / ACCUM_STEPS).backward()

            if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_ens_ld):
                torch.nn.utils.clip_grad_norm_(teacher.parameters(), 1.0)
                t_opt.step(); t_sched.step(); t_opt.zero_grad()

            correct += (torch.argmax(logits, 1) == lb).sum().item()
            total   += lb.size(0)

        # Validation
        teacher.eval(); vc = vt = 0
        with torch.no_grad():
            for batch in val_ens_ld:
                bi = batch["bert_ids"].to(device);   bm = batch["bert_mask"].to(device)
                xi = batch["xlmr_ids"].to(device);   xm = batch["xlmr_mask"].to(device)
                di = batch["deberta_ids"].to(device); dm = batch["deberta_mask"].to(device)
                lb = batch["label"].to(device)
                logits, _, _ = teacher(bi, bm, xi, xm, di, dm)
                vc += (torch.argmax(logits, 1) == lb).sum().item()
                vt += lb.size(0)

        val_acc = vc / vt
        tr_acc  = correct / total
        print(f"  T-Ep{epoch+1} | Train: {tr_acc:.4f} | Val: {val_acc:.4f}")

        if val_acc > best_t_acc:
            best_t_acc = val_acc; t_patience = 0
            torch.save(teacher.state_dict(), TEACHER_PATH)
        else:
            t_patience += 1
            if t_patience >= TEACHER_PATIENCE:
                print(f"  Teacher early stopping at epoch {epoch+1}.")
                break

    teacher.load_state_dict(torch.load(TEACHER_PATH))
    teacher.eval()
    print(f"[Fold {fold}] Teacher best val accuracy: {best_t_acc:.4f}")

    # ── STEP 2: Generate Teacher Soft Labels ──────────────────────────
    print(f"\n[Fold {fold}] STEP 2: Generating teacher soft labels...")

    train_ordered_ld    = DataLoader(train_ens_ds, batch_size=32, shuffle=False)
    teacher_logits_list = []

    with torch.no_grad():
        for batch in tqdm(train_ordered_ld, desc="Teacher inference"):
            bi = batch["bert_ids"].to(device);   bm = batch["bert_mask"].to(device)
            xi = batch["xlmr_ids"].to(device);   xm = batch["xlmr_mask"].to(device)
            di = batch["deberta_ids"].to(device); dm = batch["deberta_mask"].to(device)
            logits, _, _ = teacher(bi, bm, xi, xm, di, dm)
            teacher_logits_list.append(logits.cpu())

    teacher_logits_all = torch.cat(teacher_logits_list, dim=0)
    print(f"  Teacher logits shape: {teacher_logits_all.shape}")

    del teacher; torch.cuda.empty_cache(); gc.collect()

    # ── STEP 3: Train Student via KD ──────────────────────────────────
    print(f"\n[Fold {fold}] STEP 3: Training Student via Knowledge Distillation...")

    s_train_ds = StudentDataset(fold_train_texts, fold_train_labels, bert_tokenizer, MAX_LEN)
    s_val_ds   = StudentDataset(fold_val_texts,   fold_val_labels,   bert_tokenizer, MAX_LEN)

    kd_ds    = KDDataset(s_train_ds, teacher_logits_all)
    kd_ld    = DataLoader(kd_ds, batch_size=32, shuffle=True,
                          worker_init_fn=lambda wid: set_seed(42 + wid))
    s_val_ld = DataLoader(s_val_ds, batch_size=32)

    set_seed(42)
    student = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=num_classes
    ).to(device)
    for p in student.bert.embeddings.parameters():
        p.requires_grad = False
    student.dropout = nn.Dropout(0.3)

    s_opt     = torch.optim.AdamW(student.parameters(), lr=KD_LR, weight_decay=0.01)
    kd_total  = len(kd_ld) * KD_EPOCHS
    s_sched   = get_linear_schedule_with_warmup(s_opt,
                    num_warmup_steps=int(0.1 * kd_total),
                    num_training_steps=kd_total)

    best_s_acc   = 0.0
    s_patience   = 0
    STUDENT_PATH = f"/kaggle/working/fold{fold}_student.pt"

    for epoch in range(KD_EPOCHS):
        student.train(); correct = total = 0

        for batch in tqdm(kd_ld, desc=f"S-Ep{epoch+1}"):
            inp = batch["input_ids"].to(device)
            msk = batch["attention_mask"].to(device)
            lb  = batch["label"].to(device)
            tl  = batch["teacher_logits"].to(device)

            out  = student(input_ids=inp, attention_mask=msk)
            loss = kd_loss(out.logits, tl, lb, alpha=0.5, temperature=4.0, weight=cw)

            s_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            s_opt.step(); s_sched.step()

            correct += (torch.argmax(out.logits, 1) == lb).sum().item()
            total   += lb.size(0)

        # Validation
        student.eval()
        sv_true = []; sv_pred = []; sv_prob = []
        with torch.no_grad():
            for batch in s_val_ld:
                inp = batch["input_ids"].to(device)
                msk = batch["attention_mask"].to(device)
                lb  = batch["label"].to(device)
                out = student(input_ids=inp, attention_mask=msk)
                probs = torch.softmax(out.logits, 1)
                preds = torch.argmax(probs, 1)
                sv_true.extend(lb.cpu().numpy())
                sv_pred.extend(preds.cpu().numpy())
                sv_prob.extend(probs.cpu().numpy())

        s_val_acc = accuracy_score(sv_true, sv_pred)
        print(f"  S-Ep{epoch+1} | Train: {correct/total:.4f} | Val: {s_val_acc:.4f}")

        if s_val_acc > best_s_acc:
            best_s_acc = s_val_acc; s_patience = 0
            torch.save(student.state_dict(), STUDENT_PATH)
        else:
            s_patience += 1
            if s_patience >= KD_PATIENCE:
                print(f"  Student early stopping at epoch {epoch+1}.")
                break

    student.load_state_dict(torch.load(STUDENT_PATH))
    student.eval()

    # ── STEP 4: Final Evaluation ───────────────────────────────────────
    print(f"\n[Fold {fold}] STEP 4: Final evaluation on unseen validation set...")

    sv_true = []; sv_pred = []; sv_prob = []
    with torch.no_grad():
        for batch in s_val_ld:
            inp = batch["input_ids"].to(device)
            msk = batch["attention_mask"].to(device)
            lb  = batch["label"].to(device)
            out = student(input_ids=inp, attention_mask=msk)
            probs = torch.softmax(out.logits, 1)
            preds = torch.argmax(probs, 1)
            sv_true.extend(lb.cpu().numpy())
            sv_pred.extend(preds.cpu().numpy())
            sv_prob.extend(probs.cpu().numpy())

    sv_true = np.array(sv_true)
    sv_prob = np.array(sv_prob)

    acc  = accuracy_score(sv_true, sv_pred)
    prec = precision_score(sv_true, sv_pred, average="weighted", zero_division=0)
    rec  = recall_score(sv_true, sv_pred,    average="weighted", zero_division=0)
    f1   = f1_score(sv_true, sv_pred,        average="weighted", zero_division=0)
    auc  = roc_auc_score(sv_true, sv_prob[:, 1])

    print(f"\n  Fold {fold} Results:")
    print(f"  Accuracy  : {acc  * 100:.2f}%")
    print(f"  Precision : {prec * 100:.2f}%")
    print(f"  Recall    : {rec  * 100:.2f}%")
    print(f"  F1 Score  : {f1   * 100:.2f}%")
    print(f"  AUC       : {auc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(sv_true, sv_pred, target_names=["Non-Hate", "Hate"]))

    # Save this fold's result to disk — persists after Kaggle commit
    save_fold_result(fold, {
        "acc": float(acc), "prec": float(prec),
        "rec": float(rec), "f1":   float(f1),
        "auc": float(auc)
    })

    # Cleanup GPU memory before next fold
    del student, kd_ds, kd_ld, train_ens_ds, val_ens_ds, teacher_logits_all
    torch.cuda.empty_cache(); gc.collect()

# ============================================================
# FINAL SUMMARY — aggregate all completed folds
# ============================================================

print("\n\n" + "=" * 70)
print("FINAL AGGREGATED RESULTS")
print("=" * 70)

fold_results    = []
completed_folds = []

for fold in range(1, N_SPLITS + 1):
    result = load_fold_result(fold)
    if result is None:
        print(f"  Fold {fold} not yet completed.")
        continue
    completed_folds.append(fold)
    fold_results.append(result["metrics"])

if completed_folds:
    accs  = np.array([r["acc"]  for r in fold_results])
    precs = np.array([r["prec"] for r in fold_results])
    recs  = np.array([r["rec"]  for r in fold_results])
    f1s   = np.array([r["f1"]   for r in fold_results])
    aucs  = np.array([r["auc"]  for r in fold_results])

    print(f"\n  Completed folds: {completed_folds}\n")
    print(f"  {'Fold':<8} {'Accuracy':>10} {'Precision':>11} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    print("  " + "-" * 55)

    for fold, r in zip(completed_folds, fold_results):
        print(f"  {fold:<8} {r['acc']*100:>9.2f}% {r['prec']*100:>10.2f}% "
              f"{r['rec']*100:>7.2f}% {r['f1']*100:>7.2f}% {r['auc']:>8.4f}")

    print("  " + "-" * 55)
    print(f"  {'Mean':<8} {accs.mean()*100:>9.2f}% {precs.mean()*100:>10.2f}% "
          f"{recs.mean()*100:>7.2f}% {f1s.mean()*100:>7.2f}% {aucs.mean():>8.4f}")
    print(f"  {'Std':<8} {accs.std()*100:>9.2f}% {precs.std()*100:>10.2f}% "
          f"{recs.std()*100:>7.2f}% {f1s.std()*100:>7.2f}% {aucs.std():>8.4f}")

    if len(completed_folds) == N_SPLITS:
        print("\n\n" + "=" * 70)
        print("ALL 5 FOLDS COMPLETE — PAPER REPORT (Mean ± Std):")
        print("=" * 70)
        print(f"  Accuracy  : {accs.mean()*100:.2f}% ± {accs.std()*100:.2f}%")
        print(f"  Precision : {precs.mean()*100:.2f}% ± {precs.std()*100:.2f}%")
        print(f"  Recall    : {recs.mean()*100:.2f}% ± {recs.std()*100:.2f}%")
        print(f"  F1 Score  : {f1s.mean()*100:.2f}% ± {f1s.std()*100:.2f}%")
        print(f"  AUC       : {aucs.mean():.4f} ± {aucs.std():.4f}")

        # Save final results CSV
        pd.DataFrame([{"fold": f, **r} for f, r in zip(completed_folds, fold_results)]).to_csv(
            "/kaggle/working/cv_results_final.csv", index=False
        )
        print("\n  Results saved to /kaggle/working/cv_results_final.csv")
    else:
        remaining = [f for f in range(1, N_SPLITS + 1) if f not in completed_folds]
        print(f"\n  Remaining folds: {remaining}")
        print("  → Commit this session, then run again to continue.")

  Resumed from previous session: fold_4.json
  Resumed from previous session: fold_1.json
  Resumed from previous session: fold_2.json
  Resumed from previous session: fold_3.json
Device: cuda
Dataset size: 25570
labels
1    14565
0    11005
Name: count, dtype: int64

Loading tokenizers...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  S-Ep3 | Train: 0.9441 | Val: 0.9106


S-Ep4: 100%|██████████| 640/640 [07:32<00:00,  1.42it/s]


  S-Ep4 | Train: 0.9634 | Val: 0.9087


S-Ep5: 100%|██████████| 640/640 [07:32<00:00,  1.41it/s]


  S-Ep5 | Train: 0.9739 | Val: 0.9089
  Student early stopping at epoch 5.

[Fold 5] STEP 4: Final evaluation on unseen validation set...

  Fold 5 Results:
  Accuracy  : 91.20%
  Precision : 91.19%
  Recall    : 91.20%
  F1 Score  : 91.19%
  AUC       : 0.9662

  Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.91      0.89      0.90      2201
        Hate       0.92      0.93      0.92      2913

    accuracy                           0.91      5114
   macro avg       0.91      0.91      0.91      5114
weighted avg       0.91      0.91      0.91      5114

  Fold 5 result saved → /kaggle/working/cv_checkpoints/fold_5.json


FINAL AGGREGATED RESULTS

  Completed folds: [1, 2, 3, 4, 5]

  Fold       Accuracy   Precision   Recall       F1      AUC
  -------------------------------------------------------
  1            91.01%      91.20%   91.01%   91.03%   0.9647
  2            90.87%      91.01%   90.87%   90.89%   0.9626
  3          